# Lorenz-63 Model

The Lorenz-63 model is commonly used to assess various data assimilation methods for nonlinear systems. This model is of interest because of its nonlinear chaotic behavior and low dimensions. The simple three-variable model is composed of a system of nonlinear ordinary differential equations:
\begin{gather*}
	\frac{\partial x}{\partial t} &=& \sigma (y-x) \\
	\frac{\partial y}{\partial t} &=& \rho x-y-xz \\
	\frac{\partial z}{\partial t} &=& xy-\beta z
\end{gather*}
We will use the following common parameter values for the reference solution: $\sigma = 10$, $\rho = 28$, and $\beta = 8/3$. 

<!-- The reference solution will have the initial conditions of $(x_0, y_0, z_0) = (1.508870, -1.531271, 25.46091)$ and a time interval of $t \in [0, 40]$. -->

The observations are generated by adding Gaussian noise with zero mean and a variance of 4.0 (standard deviation of 2.0) to the reference solution and will be measured at regular time intervals $\Delta t = 0.1$.

In [1]:
# LIBRARIES
from main import run_data_assimilation
from models.lorenz63 import Lorenz63
from observation_operators import LinearGaussian
from filters import KernelDensityEstimation
from filters import EnsembleKalmanFilter

from utils.plots import plot_rmses, plot_timeplots, plot_particles, plot_sigmas

import os
import numpy as np
import scipy.io as sio
import matplotlib.pyplot as plt
import ipywidgets # NEEDED TO MAKE interactive plots
import json
import itertools


import ipywidgets as widgets
from ipywidgets import interact
from IPython.display import display


RANDOM_SD = 10
np.random.seed(RANDOM_SD) # TODO: remove

data_path = "data/initial/"
exp_path = "exp" # for subtracting predicted mean
# exp_path = "exp_old" # for data normalization
T_spinup = 2000 # NOTE now used in model parameters

# TODO:
# - Save models so we do not have to constantly rerun (Put in an override run ...)
# - Implement cross validation
# - Clean Up ABCs and how they interact with main.py

In [2]:
# Define all ensemble sizes to test
# ensemble_sizes = [10, 20, 40, 60, 100, 200, 400, 600]
ensemble_sizes = [10, 20, 40, 60, 100]

sigma_min_xs = [0.01, 0.05, 0.1, 0.2, 0.5]
sigma_ys = [1, 2]

# Filters to run (each as a dictionary)
filter_variants = [
    {"type": "KDE", "scheduler": "VE", "sigma_max": 50},
    {"type": "KDE", "scheduler": "VP", "sigma_max": 10},
    {"type": "EnKF"}
]

models = {}

In [3]:
# SET UP observation operator
obs_op_params = {
    'mean': 0,
    'std': 2,
    'random_sd': RANDOM_SD
}
observation_op = LinearGaussian(**obs_op_params)

# Model parameters that will be the same for every variation
global_model_params = {
    'sigma': 10.0,
    'rho': 28.0,
    'beta': 8.0 / 3.0,
    'for_dt': 0.05, # Process timestep
    'obs_dt': 0.1, # Observation timestep
    'T_spinup':2000,
    'T_burnin': 2000,
    'T_steps': 4000, # Number of assimilation steps
    'process_noise': 10e-2, # std PROCESS
    'random_sd': RANDOM_SD,
    'initial_time': 200, # TODO: Replace with T_spinup
    'obs_op': observation_op # std # REFERENCE 
}

# Filter parameters that are required for both KDE and EnKF filters
global_filter_params = {
    'obs_op': observation_op, 
    'random_sd': RANDOM_SD,
} 

for ensemble_size in ensemble_sizes:

    # Load initial data
    data = sio.loadmat(data_path + f"spinup_M{ensemble_size}.mat")['model']
    initial_ensemble = np.array(data['x0'][0][0])

    # SET UP model * * * * * * * * * * * * * * * * * * * * * * * * * *
    dynamic_model_params = {
        'ensemble_size': ensemble_size,
        'initial_ensemble': np.array(data['x0'][0][0]),
        'ref_initial_state': np.array(data['xt'][0][0].T)[T_spinup-1] # REF
    }

    model_params = {**global_model_params, **dynamic_model_params} # Merge dictionaries

    # Iterate through filter variants * * * * * * * * * * * * * * * * *
    for filter_config in filter_variants:
        filter_type = filter_config["type"]

        if filter_type == "KDE":
            scheduler = filter_config["scheduler"]
            sigma_max = filter_config["sigma_max"]

            # For saving models into dict
            kde_key = f"KDE {scheduler}"
            if kde_key not in models:
                models[kde_key] = {}  # Create empty sub-dictionary

            # Iterate through different sigma values
            for sigma_min_x in sigma_min_xs:
                for sigma_y in sigma_ys:

                    model = Lorenz63(**model_params) # Initialize model

                    # Set Up filter parameters
                    kde_filter_params = {
                        'scheduler': scheduler,
                        'sigma_min': sigma_min_x,
                        'sigma_max': sigma_max,  # Different for VP and VE
                        'sigma_y': sigma_y,
                        'N_tsteps': 1000  # Pseudo-time steps
                    }

                    filter_params = {**global_filter_params, **kde_filter_params}
                    filter = KernelDensityEstimation(**filter_params) # Initialize filter
                    
                    # Construct filename
                    filename = f"{str(model)}_M{ensemble_size}_KDE_{scheduler}_SX{sigma_min_x}_SY{sigma_y}.npz"

                    # Run model
                    model = run_data_assimilation(exp_path, filename, model, filter)

                    # Store in dictionary
                    models[kde_key][(ensemble_size, sigma_min_x, sigma_y)] = model

        elif filter_type == "EnKF":

            model = Lorenz63(**model_params) # Initialize model
            
            if "EnKF" not in models:
                models["EnKF"] = {}  # Create empty sub-dictionary

            # Set up EnKF filter
            enkf_filter_params = {}  # No additional parameters needed

            filter_params = {**global_filter_params, **enkf_filter_params}
            filter = EnsembleKalmanFilter(**filter_params)  # Initialize EnKF filter

            # Construct filename
            filename = f"{str(model)}_M{ensemble_size}_EnKF.npz"

            # Run model 
            model = run_data_assimilation(exp_path, filename, model, filter)

            # Store in dictionary
            models["EnKF"][ensemble_size] = model

Loading existing data from exp/L63_M10_KDE_VE_SX0.01_SY1.npz.
Loading existing data from exp/L63_M10_KDE_VE_SX0.01_SY2.npz.
Loading existing data from exp/L63_M10_KDE_VE_SX0.05_SY1.npz.
Loading existing data from exp/L63_M10_KDE_VE_SX0.05_SY2.npz.
Loading existing data from exp/L63_M10_KDE_VE_SX0.1_SY1.npz.
Loading existing data from exp/L63_M10_KDE_VE_SX0.1_SY2.npz.
Loading existing data from exp/L63_M10_KDE_VE_SX0.2_SY1.npz.
Loading existing data from exp/L63_M10_KDE_VE_SX0.2_SY2.npz.
Loading existing data from exp/L63_M10_KDE_VE_SX0.5_SY1.npz.
Loading existing data from exp/L63_M10_KDE_VE_SX0.5_SY2.npz.
Loading existing data from exp/L63_M10_KDE_VP_SX0.01_SY1.npz.
Loading existing data from exp/L63_M10_KDE_VP_SX0.01_SY2.npz.
Loading existing data from exp/L63_M10_KDE_VP_SX0.05_SY1.npz.
Loading existing data from exp/L63_M10_KDE_VP_SX0.05_SY2.npz.
Loading existing data from exp/L63_M10_KDE_VP_SX0.1_SY1.npz.
Loading existing data from exp/L63_M10_KDE_VP_SX0.1_SY2.npz.
Loading existing

DA KDE | M=20 | VP σ_x=0.5 σ_y=2:   0%|          | 0/4000 [00:00<?, ?step/s]

Saving data to exp/L63_M20_KDE_VP_SX0.5_SY2.npz.
Loading existing data from exp/L63_M20_EnKF.npz.
Loading existing data from exp/L63_M40_KDE_VE_SX0.01_SY1.npz.
Loading existing data from exp/L63_M40_KDE_VE_SX0.01_SY2.npz.
Loading existing data from exp/L63_M40_KDE_VE_SX0.05_SY1.npz.
Loading existing data from exp/L63_M40_KDE_VE_SX0.05_SY2.npz.
Loading existing data from exp/L63_M40_KDE_VE_SX0.1_SY1.npz.
Loading existing data from exp/L63_M40_KDE_VE_SX0.1_SY2.npz.
Loading existing data from exp/L63_M40_KDE_VE_SX0.2_SY1.npz.
Loading existing data from exp/L63_M40_KDE_VE_SX0.2_SY2.npz.


DA KDE | M=40 | VE σ_x=0.5 σ_y=1:   0%|          | 0/4000 [00:00<?, ?step/s]

Saving data to exp/L63_M40_KDE_VE_SX0.5_SY1.npz.


...

Saving data to exp/L63_M60_KDE_VE_SX0.5_SY2.npz.
Loading existing data from exp/L63_M60_KDE_VP_SX0.01_SY1.npz.
Loading existing data from exp/L63_M60_KDE_VP_SX0.01_SY2.npz.
Loading existing data from exp/L63_M60_KDE_VP_SX0.05_SY1.npz.
Loading existing data from exp/L63_M60_KDE_VP_SX0.05_SY2.npz.
Loading existing data from exp/L63_M60_KDE_VP_SX0.1_SY1.npz.
Loading existing data from exp/L63_M60_KDE_VP_SX0.1_SY2.npz.
Loading existing data from exp/L63_M60_KDE_VP_SX0.2_SY1.npz.
Loading existing data from exp/L63_M60_KDE_VP_SX0.2_SY2.npz.


DA KDE | M=60 | VP σ_x=0.5 σ_y=1:   0%|          | 0/4000 [00:00<?, ?step/s]

Saving data to exp/L63_M60_KDE_VP_SX0.5_SY1.npz.


DA KDE | M=60 | VP σ_x=0.5 σ_y=2:   0%|          | 0/4000 [00:00<?, ?step/s]

Saving data to exp/L63_M60_KDE_VP_SX0.5_SY2.npz.
Loading existing data from exp/L63_M60_EnKF.npz.
Loading existing data from exp/L63_M100_KDE_VE_SX0.01_SY1.npz.
Loading existing data from exp/L63_M100_KDE_VE_SX0.01_SY2.npz.
Loading existing data from exp/L63_M100_KDE_VE_SX0.05_SY1.npz.
Loading existing data from exp/L63_M100_KDE_VE_SX0.05_SY2.npz.
Loading existing data from exp/L63_M100_KDE_VE_SX0.1_SY1.npz.
Loading existing data from exp/L63_M100_KDE_VE_SX0.1_SY2.npz.
Loading existing data from exp/L63_M100_KDE_VE_SX0.2_SY1.npz.
Loading existing data from exp/L63_M100_KDE_VE_SX0.2_SY2.npz.
Loading existing data from exp/L63_M100_KDE_VE_SX0.5_SY1.npz.
Loading existing data from exp/L63_M100_KDE_VE_SX0.5_SY2.npz.
Loading existing data from exp/L63_M100_KDE_VP_SX0.01_SY1.npz.
Loading existing data from exp/L63_M100_KDE_VP_SX0.01_SY2.npz.
Loading existing data from exp/L63_M100_KDE_VP_SX0.05_SY1.npz.
Loading existing data from exp/L63_M100_KDE_VP_SX0.05_SY2.npz.
Loading existing data from

In [4]:
plot_rmses(models)

interactive(children=(Dropdown(description='σ_x', options=(0.01, 0.05, 0.1, 0.2, 0.5), value=0.01), Dropdown(d…

In [5]:
# X, Y = np.meshgrid(sigma_min_xs, sigma_ys)
# print(X)
# plot_sigmas(models)

In [6]:
plot_timeplots(models)

interactive(children=(FloatSlider(value=400.0, description='Start Time', max=595.0000000000001, min=200.0, ste…

Button(button_style='success', description='Save Plot', style=ButtonStyle())

In [7]:
plot_particles(models)

interactive(children=(FloatSlider(value=400.0, description='Start Time', max=600.000000000075, min=400.0, step…

Button(button_style='success', description='Save Plot', style=ButtonStyle())